# Phase 3 · Pair-level disease attribution

For each candidate PMID, the disease gate evaluates a specific pathogen, host, and disease relationship. A supported result requires a supporting span linked to that PMID. Keep uncertainty, insufficient evidence, and unsupported outcomes as distinct decisions.

The decision runner is resumable. Its cache key includes pair scope, source hashes, question/profile/model versions, and run configuration.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, log_step
ROOT, LOG, LOG_PATH = configure_notebook('03_disease_gate', 'pilot-v1')
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 11:25:14,060 INFO Notebook session started at 2026-11-02T11:25:14.060400+00:00


2026-11-02 11:25:14,061 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 11:25:14,061 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/03_disease_gate.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/03_disease_gate.log


## Step 1 · Set up the run ledger and inspect literature inputs

In [2]:
import pandas as pd, yaml
from IPython.display import display
from pathogen_atlas_v4.decision_runner import run_decisions
from pathogen_atlas_v4.evidence import normalize_pair_gate
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
RUN_ID = CONFIG['run']['run_id']
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
STAGING_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/'))[-1]
MATCHES_PATH = STAGING_ROOT / 'literature/pathogen_article_matches.parquet'
ARTICLES_PATH = STAGING_ROOT / 'literature/articles.parquet'
print('Run:', RUN_ID, '| matches available:', MATCHES_PATH.exists(), '| articles available:', ARTICLES_PATH.exists())
LOG.info('Decision preflight run=%s match_file=%s article_file=%s', RUN_ID, MATCHES_PATH, ARTICLES_PATH)

2026-11-02 11:25:14,482 INFO Decision preflight run=pilot-v1 match_file=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/staging/v3-to-v4-20261102-ncbi-f278697a8ba6-v3/literature/pathogen_article_matches.parquet article_file=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/staging/v3-to-v4-20261102-ncbi-f278697a8ba6-v3/literature/articles.parquet


Run: pilot-v1 | matches available: True | articles available: True


## Step 2 · Estimate the candidate work before provider calls

In [3]:
if MATCHES_PATH.exists():
    matches = pd.read_parquet(MATCHES_PATH, columns=['pathogen_id','pmid','selected_for_fetch'])
    print('Imported pathogen–article matches:', f'{len(matches):,}')
    print('Distinct imported pathogen–PMID pairs:', f'{matches.drop_duplicates(["pathogen_id","pmid"]).shape[0]:,}')
    print('Selected for text retrieval:', f'{int(matches.selected_for_fetch.fillna(False).sum()):,}')
else:
    matches = pd.DataFrame(columns=['pathogen_id','pmid','selected_for_fetch'])
    print('No literature matches are staged for this run yet.')
print('Configured host-aware pair count:', len(CONFIG['run'].get('pairs', [])))
print('Native V4 pair-gate tasks require pair-aware search results and fetched text.')
LOG.info('Preflight matches=%d configured_pairs=%d', len(matches), len(CONFIG['run'].get('pairs', [])))

Imported pathogen–article matches: 3,450,866


2026-11-02 11:25:15,297 INFO Preflight matches=3450866 configured_pairs=120


Distinct imported pathogen–PMID pairs: 3,450,866
Selected for text retrieval: 371,223
Configured host-aware pair count: 120
Native V4 pair-gate tasks require pair-aware search results and fetched text.


## Step 3 · Validate the gate record shape and evidence-span rule

In [4]:
# Synthetic schema example only; it is not appended to the release ledger.
example = normalize_pair_gate(pathogen_id='bacteria:example', host_id='ncbi:9913', pmid='12345678',
    outcome='supported', confidence=0.93, evidence_text='The isolate caused clinical mastitis in cattle.',
    title_hash='example-title-hash', abstract_hash='example-text-hash', run_id='schema-example')
display(example)
LOG.info('Validated supported gate schema with a nonempty evidence span')

{'pathogen_id': 'bacteria:example',
 'host_id': 'ncbi:9913',
 'relationship_context': 'disease',
 'pmid': '12345678',
 'title_hash': 'example-title-hash',
 'abstract_hash': 'example-text-hash',
 'evidence_span': 'The isolate caused clinical mastitis in cattle.',
 'decision': 'supported',
 'confidence': 0.93,
 'provenance_status': 'v4_native',
 'run_id': 'schema-example'}

2026-11-02 11:25:15,301 INFO Validated supported gate schema with a nonempty evidence span


## Step 4 · Prepare provider task contract

In [5]:
PAIR_TASKS = []  # populate from this run's pair-aware PMID matches and fetched title/text records
PAIR_GATE_HANDLER = None  # configure a model/provider adapter that returns outcome, confidence, evidence_span
LEDGER_PATH = RUN_DIR / 'decision_ledger.parquet'
RAW_RESPONSE_PATH = RUN_DIR / 'raw_responses.jsonl'
print('Tasks ready:', len(PAIR_TASKS), '| handler ready:', callable(PAIR_GATE_HANDLER))
print('Ledger:', LEDGER_PATH)
print('Raw responses:', RAW_RESPONSE_PATH)
LOG.info('Decision tasks prepared=%d handler_ready=%s', len(PAIR_TASKS), callable(PAIR_GATE_HANDLER))

2026-11-02 11:25:15,304 INFO Decision tasks prepared=0 handler_ready=False


Tasks ready: 0 | handler ready: False
Ledger: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/decision_ledger.parquet
Raw responses: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/raw_responses.jsonl


## Step 5 · Run or resume the disease gate

In [6]:
RUN_DECISIONS = False
MAX_ATTEMPTS = int(CONFIG['decisions'].get('max_attempts', 2))
if RUN_DECISIONS:
    if not PAIR_TASKS or not callable(PAIR_GATE_HANDLER):
        raise RuntimeError('Provide pair-scoped tasks and a configured provider handler before running decisions.')
    DECISIONS = run_decisions(PAIR_TASKS, PAIR_GATE_HANDLER, ledger_path=LEDGER_PATH,
                              raw_response_path=RAW_RESPONSE_PATH, max_attempts=MAX_ATTEMPTS)
    display(DECISIONS.groupby('status').size().rename('decisions').to_frame())
    LOG.info('Gate run complete: %s', DECISIONS.status.value_counts().to_dict())
else:
    print('Provider calls skipped; set RUN_DECISIONS=True after loading auditable pair tasks and provider credentials.')
    LOG.info('Gate run skipped: RUN_DECISIONS=False')

2026-11-02 11:25:15,307 INFO Gate run skipped: RUN_DECISIONS=False


Provider calls skipped; set RUN_DECISIONS=True after loading auditable pair tasks and provider credentials.


## Step 6 · Review outcomes and failures

In [7]:
if LEDGER_PATH.exists():
    ledger = pd.read_parquet(LEDGER_PATH)
    display(ledger.groupby(['scope','status'], dropna=False).size().rename('rows').to_frame())
    if 'error' in ledger:
        failures = ledger.loc[ledger.status.eq('failed'), ['pathogen_id','host_id','pmid','error']]
        print('Failed / retryable rows:', len(failures))
        display(failures.head(20))
else:
    print('Decision ledger not created yet:', LEDGER_PATH)
LOG.info('Review step complete; ledger_exists=%s', LEDGER_PATH.exists())

2026-11-02 11:25:15,310 INFO Review step complete; ledger_exists=False


Decision ledger not created yet: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/decision_ledger.parquet


The raw-response file and decision ledger are separate: the ledger is the normalized audit table; raw provider output is retained for review and reproducibility.